Accident Severity Prediction using PySpark ML

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

from pyspark.ml import Pipeline

from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)

from pyspark.ml.classification import RandomForestClassifier

from pyspark.ml.evaluation import MulticlassClassificationEvaluator

from pyspark.ml.functions import vector_to_array

In [2]:
spark = (
    SparkSession.builder
    .appName("Road Accident Severity Prediction")
    .master("local[*]")
    .getOrCreate()
)

26/07/30 22:25:32 WARN Utils: Your hostname, LAPTOP-S89A4J3G resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/30 22:25:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/30 22:25:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.csv(
    "../cleaned_data/road_accident_cleaned",
    header=True,
    inferSchema=True
)

df.printSchema()

root
 |-- Accident_Index: string (nullable = true)
 |-- Accident Date: date (nullable = true)
 |-- Month: string (nullable = true)
 |-- Day_of_Week: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Junction_Control: string (nullable = true)
 |-- Junction_Detail: string (nullable = true)
 |-- Accident_Severity: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Light_Conditions: string (nullable = true)
 |-- Local_Authority_(District): string (nullable = true)
 |-- Carriageway_Hazards: string (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Number_of_Casualties: integer (nullable = true)
 |-- Number_of_Vehicles: integer (nullable = true)
 |-- Police_Force: string (nullable = true)
 |-- Road_Surface_Conditions: string (nullable = true)
 |-- Road_Type: string (nullable = true)
 |-- Speed_limit: integer (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- Urban_or_Rural_Area: string (nullable = true)
 |-- Weather_Conditions: stri

In [4]:
feature_cols = [

    "Month",

    "Day_of_Week",

    "Junction_Control",

    "Junction_Detail",

    "Light_Conditions",

    "Carriageway_Hazards",

    "Number_of_Casualties",

    "Number_of_Vehicles",

    "Road_Surface_Conditions",

    "Road_Type",

    "Speed_limit",

    "Urban_or_Rural_Area",

    "Weather_Conditions",

    "Vehicle_Type"

]

target = "Accident_Severity"

ml_df = df.select(feature_cols + [target])

ml_df.show(5)

+-----+-----------+--------------------+--------------------+----------------+-------------------+--------------------+------------------+-----------------------+------------------+-----------+-------------------+------------------+------------+-----------------+
|Month|Day_of_Week|    Junction_Control|     Junction_Detail|Light_Conditions|Carriageway_Hazards|Number_of_Casualties|Number_of_Vehicles|Road_Surface_Conditions|         Road_Type|Speed_limit|Urban_or_Rural_Area|Weather_Conditions|Vehicle_Type|Accident_Severity|
+-----+-----------+--------------------+--------------------+----------------+-------------------+--------------------+------------------+-----------------------+------------------+-----------+-------------------+------------------+------------+-----------------+
|  Jun|    Tuesday|Give way or uncon...|T or staggered ju...|        Daylight|               None|                   1|                 1|                    Dry|Single carriageway|         30|              U

In [5]:
label_indexer = StringIndexer(
    inputCol="Accident_Severity",
    outputCol="label"
)

In [6]:
categorical_cols = [

    "Month",

    "Day_of_Week",

    "Junction_Control",

    "Junction_Detail",

    "Light_Conditions",

    "Carriageway_Hazards",

    "Road_Surface_Conditions",

    "Road_Type",

    "Urban_or_Rural_Area",

    "Weather_Conditions",

    "Vehicle_Type"

]

In [7]:
numeric_cols = [

    "Number_of_Casualties",

    "Number_of_Vehicles",

    "Speed_limit"

]

In [8]:
indexers = [
    StringIndexer(
        inputCol=col,
        outputCol=col + "_Index",
        handleInvalid="keep"
    )
    for col in categorical_cols
]

In [9]:
encoders = [
    OneHotEncoder(
        inputCol=col + "_Index",
        outputCol=col + "_Vec"
    )
    for col in categorical_cols
]

Create Feature Vector

In [10]:
assembler = VectorAssembler(
    inputCols=[col + "_Vec" for col in categorical_cols] + numeric_cols,
    outputCol="features"
)

Create the ML Pipeline

In [11]:
pipeline = Pipeline(
    stages=indexers + encoders + [label_indexer, assembler]
)

In [12]:
pipeline_model = pipeline.fit(ml_df)

processed_df = pipeline_model.transform(ml_df)

processed_df.select("features", "label").show(5, truncate=False)

26/07/30 22:26:55 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                   |label|
+-----------------------------------------------------------------------------------------------------------+-----+
|(83,[4,14,19,26,34,39,45,50,55,57,65,80,81,82],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,30.0]) |0.0  |
|(83,[6,12,19,26,34,39,45,50,55,57,65,80,81,82],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,30.0]) |0.0  |
|(83,[1,12,19,26,34,39,45,50,55,57,65,80,81,82],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,30.0]) |0.0  |
|(83,[10,18,21,27,34,39,46,50,55,57,65,80,81,82],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,30.0])|0.0  |
|(83,[11,13,19,26,34,39,45,50,55,57,65,80,81,82],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,30.0])|0.0  |
+-----------------------------------------------------------------------

Train-Test Split

In [13]:
train_df, test_df = processed_df.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training Records :", train_df.count())
print("Testing Records  :", test_df.count())

Training Records : 240682


Testing Records  : 59809


Random Forest Model

In [14]:
rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=100,
    maxDepth=10,
    seed=42
)

rf_model = rf.fit(train_df)

26/07/30 22:28:17 WARN DAGScheduler: Broadcasting large task binary with size 1317.8 KiB
26/07/30 22:28:52 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/07/30 22:29:11 WARN DAGScheduler: Broadcasting large task binary with size 3.8 MiB
26/07/30 22:29:32 WARN DAGScheduler: Broadcasting large task binary with size 6.3 MiB
26/07/30 22:30:00 WARN DAGScheduler: Broadcasting large task binary with size 1344.5 KiB
26/07/30 22:30:03 WARN DAGScheduler: Broadcasting large task binary with size 10.1 MiB
26/07/30 22:30:35 WARN DAGScheduler: Broadcasting large task binary with size 2004.9 KiB


Make Predictions

In [15]:
predictions = rf_model.transform(test_df)

predictions.select(
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)

26/07/30 22:30:42 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB


+-----+----------+-------------------------------------------------------------+
|label|prediction|probability                                                  |
+-----+----------+-------------------------------------------------------------+
|0.0  |0.0       |[0.8837102846027836,0.10752960558246903,0.008760109814747457]|
|1.0  |0.0       |[0.8191211837304523,0.16468134418729355,0.016197472082254142]|
|0.0  |0.0       |[0.8378385703508312,0.1507418050589433,0.01141962459022545]  |
|0.0  |0.0       |[0.8935014854733835,0.10013843143432732,0.006360083092289184]|
|0.0  |0.0       |[0.9086573787888705,0.08478389057512778,0.00655873063600173] |
|0.0  |0.0       |[0.838181295618884,0.15032949400480547,0.011489210376310616] |
|0.0  |0.0       |[0.9044241194333507,0.08990415189376257,0.005671728672886815]|
|0.0  |0.0       |[0.858800838609579,0.12931799320434811,0.011881168186072771] |
|0.0  |0.0       |[0.7950841810682313,0.17877511680163186,0.02614070213013676] |
|1.0  |0.0       |[0.8095182

Evaluate the Model

In [16]:
accuracy = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

print("Accuracy :", accuracy.evaluate(predictions))

26/07/30 22:30:46 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB


Accuracy : 0.8532829507264793


In [17]:
f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

print("F1 Score :", f1.evaluate(predictions))

26/07/30 22:30:58 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB


F1 Score : 0.7857319290776168


In [18]:
precision = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

print("Precision :", precision.evaluate(predictions))

26/07/30 22:31:07 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB


Precision : 0.7280917940004873


In [19]:
recall = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

print("Recall :", recall.evaluate(predictions))

26/07/30 22:31:18 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB


Recall : 0.8532829507264793


Confusion Matrix

In [20]:
predictions.groupBy("label", "prediction") \
    .count() \
    .orderBy("label", "prediction") \
    .show(50)

26/07/30 22:31:30 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
26/07/30 22:31:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:31:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:31:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:31:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:31:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:31:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:31:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:31:34 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will no

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|51034|
|  1.0|       0.0| 8015|
|  2.0|       0.0|  760|
+-----+----------+-----+



26/07/30 22:31:40 WARN DAGScheduler: Broadcasting large task binary with size 4.7 MiB


Improving the Model by Handling Class Imbalance

The baseline Random Forest model achieved good overall accuracy but failed to correctly classify the minority classes (Serious and Fatal accidents). To improve the model's ability to learn these classes, the training dataset is balanced using oversampling before retraining the classifier.

In [21]:
train_df.groupBy("label").count().show()

26/07/30 22:31:43 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:31:43 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:31:43 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:31:43 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:31:43 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:31:43 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:31:43 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:31:43 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:31:43 WARN RowBasedKeyValueBatch: Calling spill() on

+-----+------+
|label| count|
+-----+------+
|  0.0|205482|
|  1.0| 32068|
|  2.0|  3132|
+-----+------+



In [22]:
slight = train_df.filter(col("label") == 0)
serious = train_df.filter(col("label") == 1)
fatal = train_df.filter(col("label") == 2)

print("Slight :", slight.count())
print("Serious:", serious.count())
print("Fatal  :", fatal.count())

Slight : 205482


Serious: 32068


Fatal  : 3132


In [23]:
serious_over = serious.sample(
    withReplacement=True,
    fraction=6.0,
    seed=42
)

fatal_over = fatal.sample(
    withReplacement=True,
    fraction=65.0,
    seed=42
)

In [24]:
balanced_train = slight.union(serious_over).union(fatal_over)

In [25]:
balanced_train.groupBy("label").count().show()

26/07/30 22:32:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:32:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:32:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:32:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:32:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:32:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:32:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:32:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:32:01 WARN RowBasedKeyValueBatch: Calling spill() on

+-----+------+
|label| count|
+-----+------+
|  0.0|205482|
|  1.0|192234|
|  2.0|202884|
+-----+------+



In [26]:
rf_balanced = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=100,
    maxDepth=10,
    seed=42
)

rf_balanced_model = rf_balanced.fit(balanced_train)

26/07/30 22:32:53 WARN MemoryStore: Not enough space to cache rdd_301_25 in memory! (computed 12.3 MiB so far)
26/07/30 22:32:53 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for computing block rdd_301_31 in memory.
26/07/30 22:32:54 WARN BlockManager: Persisting block rdd_301_25 to disk instead.
26/07/30 22:32:57 WARN MemoryStore: Not enough space to cache rdd_301_31 in memory! (computed 384.0 B so far)
26/07/30 22:32:57 WARN BlockManager: Persisting block rdd_301_31 to disk instead.
26/07/30 22:33:00 WARN MemoryStore: Not enough space to cache rdd_301_0 in memory! (computed 5.4 MiB so far)
26/07/30 22:33:00 WARN MemoryStore: Not enough space to cache rdd_301_4 in memory! (computed 3.6 MiB so far)
26/07/30 22:33:00 WARN MemoryStore: Not enough space to cache rdd_301_11 in memory! (computed 3.6 MiB so far)
26/07/30 22:33:00 WARN MemoryStore: Not enough space to cache rdd_301_3 in memory! (computed 8.2 MiB so far)
26/07/30 22:33:00 WARN MemoryStore: Not eno

In [27]:
balanced_predictions = rf_balanced_model.transform(test_df)

Accuracy

In [28]:
accuracy_balanced = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

print("Balanced Accuracy :", accuracy_balanced.evaluate(balanced_predictions))

26/07/30 22:40:47 WARN DAGScheduler: Broadcasting large task binary with size 8.6 MiB


Balanced Accuracy : 0.5566219130900032


In [29]:
f1_balanced = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

print("Balanced F1 Score :", f1_balanced.evaluate(balanced_predictions))

26/07/30 22:42:44 WARN DAGScheduler: Broadcasting large task binary with size 8.6 MiB


Balanced F1 Score : 0.6469429619001498


In [30]:
precision_balanced = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

print("Balanced Precision :", precision_balanced.evaluate(balanced_predictions))

26/07/30 22:43:18 WARN DAGScheduler: Broadcasting large task binary with size 8.6 MiB


Balanced Precision : 0.7962100745722007


In [31]:
recall_balanced = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

print("Balanced Recall :", recall_balanced.evaluate(balanced_predictions))

26/07/30 22:43:39 WARN DAGScheduler: Broadcasting large task binary with size 8.6 MiB


Balanced Recall : 0.5566219130900032


In [32]:
balanced_predictions.groupBy("label", "prediction") \
    .count() \
    .orderBy("label", "prediction") \
    .show(50)

26/07/30 22:43:51 WARN DAGScheduler: Broadcasting large task binary with size 8.6 MiB
26/07/30 22:43:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:43:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:43:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:43:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:43:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:43:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:43:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/30 22:43:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will no

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|30803|
|  0.0|       1.0| 8816|
|  0.0|       2.0|11415|
|  1.0|       0.0| 3099|
|  1.0|       1.0| 2028|
|  1.0|       2.0| 2888|
|  2.0|       0.0|  188|
|  2.0|       1.0|  112|
|  2.0|       2.0|  460|
+-----+----------+-----+

